In [10]:
import os

from src.datascience.entity.config_entity import ModelEvaluationConfig

In [12]:
os.environ["MLFLOW_TRACKING_USERNAME"] = "anicetdata-cpu"


In [16]:
os.environ["MLFLOW_TRACKING_PASSWORD"] = "Azegha2002#"


In [15]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/anicetdata-cpu/datascience.mlflow"

In [19]:
os.chdir("./PythonProject")

In [20]:
%pwd

'C:\\Users\\anice\\PythonProject'

In [31]:
from dataclasses import dataclass
from pathlib import Path
@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [32]:

from src.datascience.utils.common import read_yaml, create_directories
from src.datascience.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH


class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)->ModelEvaluationConfig:
        config  = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/anicetdata-cpu/datascience.mlflow"
        )
        return model_evaluation_config


In [33]:
import os

import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib


In [39]:
from src.datascience.utils.common import save_json


class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column].values

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_uri_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run(run_name="Wine quality model evaluation"):
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            scores = {"rmse": rmse, "mae": mae, "r2": r2}

            save_json(Path(self.config.metric_file_name), scores)
            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            if tracking_uri_type_store!="file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModelAnicet")
            else:
                mlflow.sklearn.log_model(model, "model")


In [40]:
try:
    config = ConfigurationManager()
    model = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-20 07:38:00,060: INFO: common] yaml file: config\config.yaml loaded successfully]
[2026-05-20 07:38:00,061: INFO: common] yaml file: params.yaml loaded successfully]
[2026-05-20 07:38:00,064: INFO: common] yaml file: schema.yaml loaded successfully]
[2026-05-20 07:38:00,065: INFO: common] Created directory: artifacts]
[2026-05-20 07:38:00,066: INFO: common] Created directory: artifacts/model_evaluation]
[2026-05-20 07:38:00,470: INFO: common] Saved json file: artifacts\model_evaluation\metrics.json]


2026/05/20 07:38:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 07:38:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticnetModelAnicet'.
2026/05/20 07:38:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModelAnicet, version 1
Created version '1' of model 'ElasticnetModelAnicet'.


🏃 View run Wine quality model evaluation at: https://dagshub.com/anicetdata-cpu/datascience.mlflow/#/experiments/0/runs/56e1c5de6f0d4fd987f6312a70dd8cbc
🧪 View experiment at: https://dagshub.com/anicetdata-cpu/datascience.mlflow/#/experiments/0
